In [13]:
import os
import torch
import h5py
import numpy as np

from torch.utils.data import DataLoader
from pathlib import Path
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F

In [2]:
import h5py
import torch
from torch.utils.data import Dataset

class CLIPDataset(Dataset):
    def __init__(self, hdf5_file):
        self.file = h5py.File(hdf5_file, 'r')
        self.cell_embeddings = self.file['cell_embeddings']
        self.image_embeddings = self.file['image_embeddings']

    def __len__(self):
        return len(self.cell_embeddings)

    def __getitem__(self, idx):
        cell_emb = torch.tensor(self.cell_embeddings[idx], dtype=torch.float32)
        image_emb = torch.tensor(self.image_embeddings[idx], dtype=torch.float32)
        
        cell_emb = cell_emb.squeeze()
        image_emb = image_emb.squeeze()
        
        assert cell_emb.shape == image_emb.shape, f"Shape mismatch: Cell {cell_emb.shape}, Image {image_emb.shape}"
        
        return image_emb, cell_emb 

    def __del__(self):
        self.file.close()

In [3]:
dataset = CLIPDataset('data/embeddings/clip_embeddings_test.h5')
dataloader = DataLoader(dataset, batch_size=32, shuffle=True, num_workers=4)

In [9]:

class MLP(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, dropout_rate=0.5):
        super(MLP, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, output_dim)
        self.layer_norm1 = nn.LayerNorm(hidden_dim)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout_rate)

    def forward(self, x):
        x = self.layer_norm1(self.fc1(x))
        x = self.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        return x



class CLIPModel(nn.Module):
    def __init__(self, image_dim, gene_dim, hidden_dim, output_dim, dropout_rate=0.5):
        super(CLIPModel, self).__init__()
        self.image_mlp = MLP(image_dim, hidden_dim, output_dim, dropout_rate)
        self.gene_mlp = MLP(gene_dim, hidden_dim, output_dim, dropout_rate)
        self.logit_scale = nn.Parameter(torch.ones([]) * np.log(1 / 0.07))

    def forward(self, image_features, gene_features):
        image_embeddings = self.image_mlp(image_features)
        gene_embeddings = self.gene_mlp(gene_features)

        # Normalize embeddings
        image_embeddings = F.normalize(image_embeddings, dim=-1)
        gene_embeddings = F.normalize(gene_embeddings, dim=-1)

        # Scaled pairwise cosine similarities
        logit_scale = self.logit_scale.exp()
        logits_per_image = logit_scale * image_embeddings @ gene_embeddings.t()
        logits_per_gene = logits_per_image.t()

        return logits_per_image, logits_per_gene

class InfoNCELoss(nn.Module):
    def __init__(self, temperature=0.07):
        super(InfoNCELoss, self).__init__()
        self.temperature = temperature

    def forward(self, logits_per_image, logits_per_gene):
        batch_size = logits_per_image.size(0)

        targets = torch.arange(batch_size).long().to(logits_per_image.device)

        loss_img = F.cross_entropy(logits_per_image / self.temperature, targets)
        loss_gene = F.cross_entropy(logits_per_gene / self.temperature, targets)

        return (loss_img + loss_gene) / 2


In [17]:
def train_clip(model, dataloader, optimizer, device, temperature=0.07):
    model.to(device)
    model.train()
    criterion = InfoNCELoss(temperature)
    total_loss = 0
    num_batches = len(dataloader)

    progress_bar = tqdm(dataloader, total=num_batches, desc="Training")

    for image_batch, gene_batch in progress_bar:
        image_batch = image_batch.to(device)
        gene_batch = gene_batch.to(device)

        optimizer.zero_grad()
        logits_per_image, logits_per_gene = model(image_batch, gene_batch)
        loss = criterion(logits_per_image, logits_per_gene)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        progress_bar.set_postfix({"Loss": f"{loss.item():.4f}"})

    average_loss = total_loss / num_batches
    progress_bar.set_postfix({"Avg Loss": f"{average_loss:.4f}"})
    progress_bar.close()

    return average_loss

In [18]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

In [19]:

image_dim = 512 
gene_dim = 512 
hidden_dim = 32
output_dim = 128

model = CLIPModel(image_dim, gene_dim, hidden_dim, output_dim).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)



In [22]:
from torchsummary import summary
summary(model, [(image_dim,), (gene_dim,)])

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Linear-1                   [-1, 32]          16,416
         LayerNorm-2                   [-1, 32]              64
              ReLU-3                   [-1, 32]               0
           Dropout-4                   [-1, 32]               0
            Linear-5                  [-1, 128]           4,224
               MLP-6                  [-1, 128]               0
            Linear-7                   [-1, 32]          16,416
         LayerNorm-8                   [-1, 32]              64
              ReLU-9                   [-1, 32]               0
          Dropout-10                   [-1, 32]               0
           Linear-11                  [-1, 128]           4,224
              MLP-12                  [-1, 128]               0
Total params: 41,408
Trainable params: 41,408
Non-trainable params: 0
---------------------------------

In [23]:

num_epochs = 10
for epoch in range(num_epochs):
    loss = train_clip(model, dataloader, optimizer, device)
    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {loss:.4f}")

Training:   0%|          | 0/11434 [00:00<?, ?it/s]

Training: 100%|██████████| 11434/11434 [01:54<00:00, 99.76it/s, Loss=2.9984] 


Epoch 1/10, Loss: 3.3483


Training: 100%|██████████| 11434/11434 [00:53<00:00, 211.83it/s, Loss=2.7373]


Epoch 2/10, Loss: 3.0271


...

Epoch 8/10, Loss: 2.6005


Training: 100%|██████████| 11434/11434 [00:52<00:00, 219.00it/s, Loss=2.1679]


Epoch 9/10, Loss: 2.5845


Training: 100%|██████████| 11434/11434 [00:52<00:00, 219.80it/s, Loss=2.4756]

Epoch 10/10, Loss: 2.5714
